In [15]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [16]:
from daemon_analysis_tools.file_handling import load_and_process_csv
from daemon_analysis_tools.data.utils import group_questions_by_journal
from daemon_analysis_tools.data.publisher import Publisher
from daemon_analysis_tools.file_handling import (
    save_answers_to_yaml,
    load_answers_from_yaml,
)

Load and process data:
- Group answers by publisher and journal, trying to uniform names written in slightly different ways.
- Store in a DataFrame

In [17]:
data = load_and_process_csv("../../data/raw/rdp.csv")

Get a `dict` labeled by publisher names of `dict`s labeled by journal names of `dict`s of `Question` instances. The `.answer` attribute contains the answers given by the respondents and the explanations text to motivate it.

In [18]:
grouped_questions = group_questions_by_journal(data)

## Resolve discrepancies

The `Question` class has a `.resolve_discrepancies` method which updates `Question.anwsers` with the correct answer.

For example, let's consider IOP's 2D Materials. Question 7 has discrepancies.

In [19]:
for journal, data in grouped_questions["IEEE"].items():
    print(journal)
    for question, answer in data.items():
        if answer.has_discrepancies():
            answer.print_qa()
    print("\n\n")

ieee_access
2. Data availability statement
  Resp. 0:
    Answer: Not mentioned in the RDP.
    Explanation: N.A.
  Resp. 1:
    Answer: Mentioned in the RDP but optional.
    Explanation: In research reproducibility: All IEEE authors are encouraged to share their data, code, and other research outputs to facilitate verification and reproducibility of experiments and their conclusions.
8. Recommended data sharing method
  Resp. 0:
    Answer: Public online repositories recommended in RDP.
    Explanation: Improve the discoverability of your data by hosting it in an easily accessible repository such as IEEE DataPort™, an online data repository of datasets and data analysis tools. IEEE DataPort accepts all types of datasets up to 2TB and provides a Digital Object Identifier (DOI) for easy citation. Standard (i.e., non-Open Access) datasets can be uploaded for free at IEEE DataPort. Articles in the IEEE Xplore® Digital Library with linked data in IEEE DataPort will have a Code & Datasets 

Inconsistencies can be removed manually, passing the index of the correct respondent.

In [20]:
for j in ["ieee_access", "ieee_electron_device_letters", "ieee_journal_of_photovoltaics", 
          "ieee_journal_of_selected_topics_in_quantum_electronics", "ieee_photonics_journal",
          "ieee_robotics_and_automation_letters", "ieee_sensors_journal", "ieee_sensors_letters",
          "ieee_transactions_on_applied_superconductivity", "ieee_transactions_on_dielectrics_and_electrical_insulation",
          "ieee_transactions_on_electron_devices", "ieee_transactions_on_industrial_electronics",
          "ieee_transactions_on_transportation_electrification", 
          "ieee_transactions_on_ultrasonics_ferroelectrics_and_frequency_control",
          "journal_of_lightwave_technology", ]:
    for i in [2, 8]:
        grouped_questions["IEEE"][j][i].resolve_discrepancy(
            correct_answer=1,
            discrepancy_reason="Text not found",
        )
    
    for i in [11, 12]:
        grouped_questions["IEEE"][j][i].resolve_discrepancy(
            correct_answer=1,
            discrepancy_reason="Formatting",
        )

Text not found
Text not found
Formatting
Formatting
Text not found
Text not found
Formatting
Formatting
Text not found
Text not found
Formatting
Formatting
Text not found
Text not found
Formatting
Formatting
Text not found
Text not found
Formatting
Formatting
Text not found
Text not found
Formatting
Formatting
Text not found
Text not found
Formatting
Formatting
Text not found
Text not found
Formatting
Formatting
Text not found
Text not found
Formatting
Formatting
Text not found
Text not found
Formatting
Formatting
Text not found
Text not found
Formatting
Formatting
Text not found
Text not found
Formatting
Formatting
Text not found
Text not found
Formatting
Formatting
Text not found
Text not found
Formatting
Formatting
Text not found
Text not found
Formatting
Formatting


In [21]:
for journal, data in grouped_questions["IEEE"].items():
    print("#############################################################")
    print(journal)
    for question, answer in data.items():
        if answer.has_discrepancies() and answer.correct_answer is None:
            answer.print_qa()

#############################################################
ieee_access
#############################################################
ieee_electron_device_letters
#############################################################
ieee_journal_of_photovoltaics
#############################################################
ieee_journal_of_selected_topics_in_quantum_electronics
#############################################################
ieee_photonics_journal
#############################################################
ieee_robotics_and_automation_letters
#############################################################
ieee_sensors_journal
#############################################################
ieee_sensors_letters
#############################################################
ieee_transactions_on_applied_superconductivity
#############################################################
ieee_transactions_on_dielectrics_and_electrical_insulation
##############################################

In [22]:
save_answers_to_yaml(
    grouped_questions,
    parent_folder="../../data/processed/all_answers",
    save_only=["IEEE"],
)

IEEE/ieee_access.yaml already exists. No data was written to prevent overwriting files modified by users. Manually delete these files if necessary.
IEEE/ieee_electron_device_letters.yaml already exists. No data was written to prevent overwriting files modified by users. Manually delete these files if necessary.
IEEE/ieee_journal_of_photovoltaics.yaml already exists. No data was written to prevent overwriting files modified by users. Manually delete these files if necessary.
IEEE/ieee_journal_of_selected_topics_in_quantum_electronics.yaml already exists. No data was written to prevent overwriting files modified by users. Manually delete these files if necessary.
IEEE/ieee_photonics_journal.yaml already exists. No data was written to prevent overwriting files modified by users. Manually delete these files if necessary.
IEEE/ieee_robotics_and_automation_letters.yaml already exists. No data was written to prevent overwriting files modified by users. Manually delete these files if necessary

After doing this, the `.get_final_answer()` method returns the correct answer.